In [ ]:
# Darstellung anpassen

from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))



In [ ]:
# Lade benötigte Programmpakete

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
import sklearn as sk
import joblib


In [ ]:
from sklearn.datasets import fetch_openml # Lade den MNIST_784-Datensatz
mnist=fetch_openml('mnist_784',as_frame=False,parser='auto')

In [ ]:
X=mnist['data'] # Features, 
y=mnist['target'] # Labels
print(f"Shape of X: {X.shape}")
# Teile in Trainings und Test-Daten. Verwende die ersten 60000 Bilder zum trainieren und die letzten 10000 zum Testen
X_train=X[:60000,:]
y_train=y[:60000]
X_test=X[60000:,:]
y_test=y[60000:]



In [ ]:
# Plotte Beispielbilder der Ziffern
fig,ax=plt.subplots(5,5,figsize=(10,10))
for i in range(25):
    j=i%5
    k=i//5

    test_image=X[i].reshape(28,28)
    ax[j,k].imshow(test_image,cmap='gray',vmin=0, vmax=255)

##### Entwickeln Sie einen Classifier der die Ziffer '5' erkennt. Verwenden Sie zur Entwicklung den Trainingsdatensatz X_train, y_train und verwenden Sie die in der LV besprochenen Methoden um ein optimales Modell mit optimalen Hyperparametern zu entwickeln. Erst wenn Sie fertig sind, und überzeugt das bestmögliche Modell gefunden zu haben, Werten Sie ihre Daten am Test-Datensatz aus. 

In [ ]:
# Preprocsessing
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
print(scaler.fit(X_train))
print(scaler.data_max_)
print(scaler.transform(X_train))
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# basic training
from sklearn.svm import SVC
# Creating a support vector classifier
svc=SVC(C=10) # probability=True

svc.fit(X_train_scaled, y_train)
joblib.dump(svc, "models/svc_model.joblib")

In [ ]:

from sklearn.model_selection import GridSearchCV, KFold
# Creating a support vector classifier
svc=SVC() # probability=True

gammas = np.logspace(1,3, 5)
cs = np.logspace(1,3, 5)
params = {'C' : cs, 'gamma': gammas}

gs = GridSearchCV(svc, params, cv=KFold(n_splits=5), scoring = "neg_mean_squared_error", return_train_score=True, verbose=3, n_jobs=-1)
gs.fit(X_train_scaled, y_train)

#svc.fit(X_train_scaled, y_train)
print(gs.best_params_)
print("Accuracy on training set: {:.3f}".format(gs.score(X_train_scaled, y_train)))
print("Accuracy on training set: {:.3f}".format(gs.best_score_))
#print("Accuracy on test set: {:.3f}".format(svc.score(X_test_scaled, y_test)))

In [ ]:
svc = joblib.load("models/svc_model.joblib")

In [ ]:
# 40 seconds
from joblib import Parallel, delayed

def batch_predict(model, X_batch):
    return model.predict(X_batch)

# Split data into 4 chunks
X_chunks = np.array_split(X_test_scaled, 4)

# Predict in parallel
predictions = Parallel(n_jobs=4)(delayed(batch_predict)(svc, chunk) for chunk in X_chunks)

# Concatenate results
y_pred = np.concatenate(predictions)
accuracy = (y_pred == y_test).mean()
print("Accuracy on test set:", accuracy)

In [ ]:
# 2min 50s
X_chunks = np.array_split(X_train_scaled, 4)

# Predict in parallel
predictions = Parallel(n_jobs=4)(delayed(batch_predict)(svc, chunk) for chunk in X_chunks)

# Concatenate results
y_pred_train = np.concatenate(predictions)
accuracy = (y_pred_train == y_train).mean()
print("Accuracy on train set:", accuracy)